In [1]:
import jax
import jax.numpy as jnp
import matplotlib.pyplot as plt
import numpy as np

from flax import nnx


In [9]:
class MLP(nnx.Module, experimental_pytree=True):
    def __init__(self, dims,rngs,* ,activation=jax.nn.relu,  activate_final=False, **kwargs):
        self.layers = [nnx.Linear(dims[i],dims[i+1],rngs=rngs, **kwargs) for i in range(len(dims)-1)]
        self.activation = activation
        self.activate_final = activate_final

    def __call__(self, x):
        for layer in self.layers[:-1]:
            x = self.activation(layer(x))
        
        out = self.layers[-1](x)
        if self.activate_final:
            out = self.activation(out)
        return out

In [10]:
m = MLP([1,50,50,1],rngs=nnx.Rngs(0))

In [11]:
m(jnp.ones((1,)))

Array([0.9715742], dtype=float32)

In [16]:

from typing import Callable, Optional, Sequence

class CouplingMLP(nnx.Module, experimental_pytree=True):

    def __init__(
        self,
        split_index: int,
        input_dim: int,
        bijector: Callable,
        rngs,
        *,
        context_dim: Optional[int] = None,
        hidden_dims: Sequence[int] = [50, 50],
        **kwargs,
    ):
        self.context_dim = context_dim if context_dim else 0
        in_dim = split_index
        dims = [in_dim + self.context_dim] + list(hidden_dims) + [in_dim]
        self.conditionor = MLP(dims, rngs=rngs, **kwargs)
        self.split_index = split_index
        self.bijector = bijector


    def __call__(self, x, context=None):
        x1, x2 = jnp.split(x, [self.split_index], axis=-1)
        if context is not None:
            if self.context_dim != context.shape[-1]:
                raise ValueError(
                    f"Context dimension mismatch. Expected {self.context_dim}, got {context.shape[-1]}."
                )
            x1 = jnp.concatenate([x1, context], axis=-1)
        condition = self.conditionor(x1)
        y2 = x2 + self.bijector(x1, condition)
        return jnp.concatenate([x1, y2], axis=-1)

In [17]:
m = CouplingMLP(1, 2, lambda x,y: x + y, rngs=nnx.Rngs(0))

In [19]:
m(jnp.ones((1,2)))

Array([[1.       , 2.9715743]], dtype=float32)

In [ ]:

class CouplingMLP(hk.Module):
    def __init__(
        self,
        split_index: int,
        bijector: Callable[[Array, Array], Array],
        num_bijector_params: int,
        context_dim: int | None,
        hidden_dims: List[int] = [
            50,
        ],
        name: str = "coupling_mlp",
        **kwargs,
    ):
        """This is a invertible MLP that splits the input into two parts and applies a bijector to the second part. The parameters of the bijector are conditioned on the first part of the input.

        Args:
            split_index (int): Where to split the array into two parts.
            bijector (Callable[[Array, Array], Array]): A bijector f: params, x -> y that takes in the parameters and the input and returns the transformed input.
            num_bijector_params (int): The number of paramters the bijector takes in.
            context (Array | None, optional): The context. Defaults to None.
            hidden_dims (List[int], optional): Hidden dimensions. Defaults to [ 50, ].
            name (str, optional): Name. Defaults to "coupling_mlp".
        """
        super().__init__(name=name)
        self.split_index = split_index
        self.context_dim = context_dim if context_dim is not None else 0
        self.bijector = bijector
        self.num_bijector_params = num_bijector_params
        self._hidden_dims = hidden_dims
        self._mlp_params = kwargs

    def __call__(self, x: Array, context: Array) -> Array:
        conditionor = hk.nets.MLP(
            [self.split_index + self.context_dim]
            + self._hidden_dims
            + [self.num_bijector_params],
            **self._mlp_params,
        )
        x1, x2 = jnp.split(x, [self.split_index], axis=-1)
        y1 = x1
        if context is not None:
            x1 = jnp.hstack([x1, context])
        params = conditionor(x1)
        y2 = self.bijector(params, x2)

        y = jnp.concatenate([y1, y2], axis=-1)
        return y
